# CQI ML Robustness Pipeline with PySpark (New AWS, 8 Core Nodes)

**MACS 30123 Final Project**


The workflow is intentionally split into two execution blocks:

1. **Smoke Run**: small window around the ChatGPT cutoff.
2. **Full Run**: full final-analysis window from 2019-01 to 2025-03.

Target logic:
- If `AcceptedAnswerId` exists, predict whether a question receives an accepted answer.
- If `AcceptedAnswerId` is unavailable, fall back to predicting whether a question belongs to the post-ChatGPT period.


In [ ]:
from pathlib import Path

AWS_ACCESS_KEY_ID = "PASTE_YOUR_ACCESS_KEY_ID_HERE"
AWS_SECRET_ACCESS_KEY = "PASTE_YOUR_SECRET_ACCESS_KEY_HERE"
AWS_SESSION_TOKEN = "PASTE_YOUR_SESSION_TOKEN_HERE"  # AWS Academy usually requires this. Leave empty if not needed.
AWS_DEFAULT_REGION = "us-east-1"

aws_dir = Path.home() / ".aws"
aws_dir.mkdir(mode=0o700, exist_ok=True)

credentials_lines = [
    "[default]",
    f"aws_access_key_id = {AWS_ACCESS_KEY_ID}",
    f"aws_secret_access_key = {AWS_SECRET_ACCESS_KEY}",
]
if AWS_SESSION_TOKEN and not AWS_SESSION_TOKEN.startswith("PASTE_"):
    credentials_lines.append(f"aws_session_token = {AWS_SESSION_TOKEN}")

(aws_dir / "credentials").write_text("\n".join(credentials_lines) + "\n")
(aws_dir / "config").write_text(f"[default]\nregion = {AWS_DEFAULT_REGION}\noutput = json\n")

print(f"Wrote AWS CLI config to {aws_dir}")
print("Next verification command:")
print("!aws sts get-caller-identity")


In [14]:
%%configure -f
{
    "conf": {
        "spark.pyspark.python": "python3",
        "spark.pyspark.virtualenv.enabled": "true",
        "spark.pyspark.virtualenv.type": "native",
        "spark.pyspark.virtualenv.bin.path": "/usr/bin/virtualenv"
    }
}


Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
1,application_1779914636069_0003,pyspark,idle,Link,Link,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
1,application_1779914636069_0003,pyspark,idle,Link,Link,✔


In [15]:
# ── Imports and EMR runtime check ────────────────────────────────────────────
# On EMR PySpark kernel, `spark` and `sc` are already provided.
# getOrCreate() reuses the existing session; creates one only if missing.
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, Bucketizer
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

spark = SparkSession.builder.appName("SO_CQI_ML_PIPELINE_NEWAWS_8CORE").getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("WARN")

EMR_PRIMARY_NODES = 1
EMR_CORE_NODES = 8
EMR_INSTANCE_TYPE = "m5.xlarge"

print("Spark version:", spark.version)
print("Default parallelism:", sc.defaultParallelism)
print(f"EMR new-AWS ML cluster: {EMR_PRIMARY_NODES} primary + {EMR_CORE_NODES} core nodes ({EMR_INSTANCE_TYPE})")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Spark version: 3.0.1-amzn-0
Default parallelism: 32
EMR new-AWS ML cluster: 1 primary + 8 core nodes (m5.xlarge)

In [16]:
# ── Shared paths and constants ───────────────────────────────────────────────
BUCKET = "macs30123-final-pj-yanjing-li-2026"
S3_POSTS_PATH = f"s3://{BUCKET}/posts_cqi_with_tier/"
S3_OUTPUT_BASE = f"s3://{BUCKET}/outputs/ml_pipeline/"
CUTOFF = "2022-12-01"

print("Input path :", S3_POSTS_PATH)
print("Output base:", S3_OUTPUT_BASE)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Input path : s3://macs30123-final-pj-yanjing-li-2026/posts_cqi_with_tier/
Output base: s3://macs30123-final-pj-yanjing-li-2026/outputs/ml_pipeline/

In [17]:
# ── Shared data-prep function ────────────────────────────────────────────────
def load_and_prepare_ml_data(start_month, end_month):
    # Load curated parquet, create features, and choose the ML label.
    raw = spark.read.parquet(S3_POSTS_PATH)
    print("Available columns:")
    print(raw.columns)

    requested_cols = [
        "Id", "CreationDate", "Score", "AnswerCount", "AcceptedAnswerId",
        "Tags", "ClosedDate", "is_python", "is_duplicate", "code_block_count",
        "primary_code_len", "cyclomatic_complexity", "halstead_volume",
        "is_parseable", "has_code", "cqi_version_a", "Reputation",
        "experience_tier", "year_month",
    ]
    available_cols = [c for c in requested_cols if c in raw.columns]

    df = raw.select(*available_cols)
    df = df.filter(
        (F.col("year_month").cast("string") >= F.lit(start_month)) &
        (F.col("year_month").cast("string") <= F.lit(end_month))
    )

    print(f"Rows before ML cleaning ({start_month} to {end_month}): {df.count():,}")
    print("Partitions:", df.rdd.getNumPartitions())

    df = (df
        .withColumn("year_month_str", F.col("year_month").cast("string"))
        .withColumn("month", F.to_date(F.concat(F.col("year_month_str"), F.lit("-01"))))
        .withColumn("months_since_cutoff", F.months_between(F.col("month"), F.to_date(F.lit(CUTOFF))))
        .withColumn("post_chatgpt", F.when(F.col("month") >= F.to_date(F.lit(CUTOFF)), 1).otherwise(0).cast("integer"))
        .withColumn("cqi_version_a", F.col("cqi_version_a").cast("double"))
        .withColumn("reputation_num", F.col("Reputation").cast("double"))
        .withColumn("code_block_count_num", F.col("code_block_count").cast("double"))
        .withColumn("primary_code_len_num", F.col("primary_code_len").cast("double"))
        .withColumn("cyclomatic_complexity_num", F.col("cyclomatic_complexity").cast("double"))
        .withColumn("halstead_volume_num", F.col("halstead_volume").cast("double"))
        .withColumn("has_code_int", F.col("has_code").cast("integer"))
        .withColumn("is_parseable_int", F.col("is_parseable").cast("integer"))
        .withColumn("is_duplicate_int", F.col("is_duplicate").cast("integer"))
    )

    df = df.withColumn(
        "tag_category",
        F.when(F.col("Tags").contains("<pandas>") | F.col("Tags").contains("<numpy>") |
               F.col("Tags").contains("<scikit-learn>") | F.col("Tags").contains("<matplotlib>"), "data_science")
         .when(F.col("Tags").contains("<list>") | F.col("Tags").contains("<dictionary>") |
               F.col("Tags").contains("<string>") | F.col("Tags").contains("<loops>"), "python_basics")
         .when(F.col("Tags").contains("<django>") | F.col("Tags").contains("<flask>") |
               F.col("Tags").contains("<fastapi>"), "web_framework")
         .when(F.col("Tags").contains("<multiprocessing>") | F.col("Tags").contains("<asyncio>") |
               F.col("Tags").contains("<ctypes>"), "systems")
         .when(F.col("Tags").contains("<nlp>") | F.col("Tags").contains("<openai-api>") |
               F.col("Tags").contains("<langchain>") | F.col("Tags").contains("<pytorch>") |
               F.col("Tags").contains("<tensorflow>"), "ml_ai")
         .otherwise("other")
    )

    df = (df
        .withColumn("log_reputation", F.log1p(F.greatest(F.col("reputation_num"), F.lit(0.0))))
        .withColumn("log_code_len", F.log1p(F.greatest(F.col("primary_code_len_num"), F.lit(0.0))))
        .withColumn("log_halstead", F.log1p(F.greatest(F.col("halstead_volume_num"), F.lit(0.0))))
    )

    if "AcceptedAnswerId" in df.columns:
        target_mode = "accepted_answer"
        df = df.withColumn("label", F.when(F.col("AcceptedAnswerId").isNotNull(), 1.0).otherwise(0.0))
        target_description = "has accepted answer"
    else:
        target_mode = "post_chatgpt"
        df = df.withColumn("label", F.col("post_chatgpt").cast("double"))
        target_description = "post-ChatGPT period"

    fill_values = {
        "log_reputation": 0.0,
        "code_block_count_num": 0.0,
        "primary_code_len_num": 0.0,
        "log_code_len": 0.0,
        "cyclomatic_complexity_num": 0.0,
        "halstead_volume_num": 0.0,
        "log_halstead": 0.0,
        "has_code_int": 0,
        "is_parseable_int": 0,
        "is_duplicate_int": 0,
    }
    df = df.fillna(fill_values)

    required = [
        "label", "cqi_version_a", "experience_tier", "tag_category",
        "log_reputation", "code_block_count_num", "primary_code_len_num",
        "log_code_len", "has_code_int", "is_parseable_int", "is_duplicate_int",
    ]
    if target_mode == "accepted_answer":
        required += ["post_chatgpt", "months_since_cutoff"]

    df = df.na.drop(subset=required)

    print("ML target mode:", target_mode)
    print("Target description:", target_description)
    print(f"Clean ML rows: {df.count():,}")
    df.groupBy("label").count().orderBy("label").show()
    df.select("Id", "year_month_str", "label", "cqi_version_a", "experience_tier", "tag_category", "post_chatgpt").show(5, truncate=False)

    return df, target_mode


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [21]:
# ── Shared pipeline builder ──────────────────────────────────────────────────
def build_ml_pipeline(target_mode):
    tier_indexer = StringIndexer(inputCol="experience_tier", outputCol="experience_tier_idx", handleInvalid="keep")
    cat_indexer = StringIndexer(inputCol="tag_category", outputCol="tag_category_idx", handleInvalid="keep")
    encoder = OneHotEncoder(
        inputCols=["experience_tier_idx", "tag_category_idx"],
        outputCols=["experience_tier_ohe", "tag_category_ohe"],
    )
    code_len_bucketizer = Bucketizer(
        splits=[0.0, 1.0, 25.0, 100.0, 500.0, 2000.0, float("inf")],
        inputCol="primary_code_len_num",
        outputCol="primary_code_len_bucket",
        handleInvalid="keep",
    )

    numeric_features = [
        "cqi_version_a", "log_reputation", "code_block_count_num", "log_code_len",
        "cyclomatic_complexity_num", "log_halstead", "has_code_int",
        "is_parseable_int", "is_duplicate_int", "primary_code_len_bucket",
    ]
    if target_mode == "accepted_answer":
        numeric_features += ["post_chatgpt", "months_since_cutoff"]

    assembler = VectorAssembler(
        inputCols=numeric_features + ["experience_tier_ohe", "tag_category_ohe"],
        outputCol="features",
        handleInvalid="skip",
    )
    lr = LogisticRegression(
        featuresCol="features",
        labelCol="label",
        predictionCol="prediction",
        probabilityCol="probability",
        maxIter=50,
    )
    pipeline = Pipeline(stages=[tier_indexer, cat_indexer, encoder, code_len_bucketizer, assembler, lr])

    print("Target mode:", target_mode)
    print("Numeric features:", numeric_features)
    print("Pipeline stages:", [type(s).__name__ for s in pipeline.getStages()])
    return pipeline, lr


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [18]:
# ── Shared training/evaluation function ──────────────────────────────────────
def train_evaluate_and_write(df, target_mode, run_label, start_month, end_month, cv_folds, param_reg, param_enet):
    s3_output_path = f"{S3_OUTPUT_BASE}{run_label}/"

    pipeline, lr = build_ml_pipeline(target_mode)
    train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)
    n_train = train_df.count()
    n_test = test_df.count()
    print(f"Train: {n_train:,}   Test: {n_test:,}")

    paramGrid = (ParamGridBuilder()
        .addGrid(lr.regParam, param_reg)
        .addGrid(lr.elasticNetParam, param_enet)
        .build())

    auc_evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
    crossval = CrossValidator(
        estimator=pipeline,
        estimatorParamMaps=paramGrid,
        evaluator=auc_evaluator,
        numFolds=cv_folds,
        parallelism=EMR_CORE_NODES,
        seed=42,
    )

    print(f"Fitting CrossValidator on new AWS EMR cluster ({EMR_CORE_NODES} core nodes)...")
    print("Run label:", run_label)
    print("Param grid size:", len(paramGrid))
    cv_model = crossval.fit(train_df)

    predictions = cv_model.transform(test_df)
    test_auc = auc_evaluator.evaluate(predictions)
    accuracy_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
    f1_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
    test_accuracy = accuracy_evaluator.evaluate(predictions)
    test_f1 = f1_evaluator.evaluate(predictions)

    best_lr = cv_model.bestModel.stages[-1]
    print(f"Test AUC     : {test_auc:.4f}")
    print(f"Test Accuracy: {test_accuracy:.4f}")
    print(f"Test F1      : {test_f1:.4f}")
    print(f"Best regParam       : {best_lr._java_obj.getRegParam()}")
    print(f"Best elasticNetParam: {best_lr._java_obj.getElasticNetParam()}")
    print(f"Intercept           : {best_lr.intercept:.4f}")

    predictions.select("label", "prediction", "probability", "cqi_version_a", "experience_tier", "tag_category").show(10, truncate=False)

    confusion = predictions.groupBy("label", "prediction").count().orderBy("label", "prediction")
    confusion.show()

    metrics_rows = [(
        run_label, target_mode, start_month, end_month,
        float(test_auc), float(test_accuracy), float(test_f1),
        float(best_lr._java_obj.getRegParam()), float(best_lr._java_obj.getElasticNetParam()),
        float(best_lr.intercept), int(n_train), int(n_test)
    )]
    metrics_df = spark.createDataFrame(
        metrics_rows,
        ["run_label", "target_mode", "start_month", "end_month", "test_auc", "test_accuracy", "test_f1",
         "best_regParam", "best_elasticNetParam", "intercept", "n_train", "n_test"]
    )
    metrics_df.show(truncate=False)

    feature_meta = predictions.schema["features"].metadata.get("ml_attr", {}).get("attrs", {})
    attrs = []
    for group in ["numeric", "binary"]:
        attrs.extend(feature_meta.get(group, []))
    feature_names = [a["name"] for a in sorted(attrs, key=lambda x: x["idx"])]
    if len(feature_names) != len(best_lr.coefficients):
        feature_names = [f"feature_{i}" for i in range(len(best_lr.coefficients))]

    coef_rows = [(feature_names[i], float(coef), abs(float(coef))) for i, coef in enumerate(best_lr.coefficients)]
    coef_rows = sorted(coef_rows, key=lambda row: row[2], reverse=True)
    coef_df = spark.createDataFrame(coef_rows, ["feature", "coefficient", "abs_coefficient"])
    coef_df.show(50, truncate=False)
    
    sample_base = predictions.withColumn(
    "probability_str",
    F.col("probability").cast("string"))

    sample_cols = [
        "Id", "year_month_str", "label", "prediction", "probability_str", "cqi_version_a",
        "experience_tier", "tag_category", "log_reputation", "code_block_count_num", "log_code_len",
    ]
    sample_pred = sample_base.select(*[c for c in sample_cols if c in sample_base.columns]).limit(100)
    sample_pred.show(20, truncate=False)
    

    metrics_df.coalesce(1).write.mode("overwrite").option("header", True).csv(s3_output_path + "metrics_csv")
    confusion.coalesce(1).write.mode("overwrite").option("header", True).csv(s3_output_path + "confusion_matrix_csv")
    coef_df.coalesce(1).write.mode("overwrite").option("header", True).csv(s3_output_path + "coefficients_csv")
    sample_pred.coalesce(1).write.mode("overwrite").option("header", True).csv(s3_output_path + "prediction_sample_csv")

    print("Wrote metrics to:", s3_output_path + "metrics_csv")
    print("Wrote confusion matrix to:", s3_output_path + "confusion_matrix_csv")
    print("Wrote coefficients to:", s3_output_path + "coefficients_csv")
    print("Wrote prediction sample to:", s3_output_path + "prediction_sample_csv")
    print("List output with:")
    print(f"aws s3 ls {s3_output_path} --recursive --summarize")

    return cv_model, predictions, metrics_df, confusion, coef_df


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [23]:
# ── Unsupervised KMeans clustering helper ────────────────────────────────────
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.feature import VectorAssembler, StandardScaler

def run_kmeans_and_write(df, run_label, start_month, end_month, k=4):
    s3_output_path = f"{S3_OUTPUT_BASE}{run_label}_kmeans/"

    clustering_features = [
        "cqi_version_a",
        "log_reputation",
        "code_block_count_num",
        "log_code_len",
        "cyclomatic_complexity_num",
        "log_halstead",
        "has_code_int",
        "is_parseable_int",
        "is_duplicate_int",
        "post_chatgpt",
        "months_since_cutoff",
    ]

    clustering_features = [c for c in clustering_features if c in df.columns]

    clean_df = df.na.drop(subset=clustering_features).cache()
    print(f"KMeans run label: {run_label}")
    print(f"Rows for clustering: {clean_df.count():,}")
    print("Features:", clustering_features)

    assembler = VectorAssembler(
        inputCols=clustering_features,
        outputCol="raw_cluster_features",
        handleInvalid="skip",
    )

    scaler = StandardScaler(
        inputCol="raw_cluster_features",
        outputCol="cluster_features",
        withMean=True,
        withStd=True,
    )

    assembled = assembler.transform(clean_df)
    scaler_model = scaler.fit(assembled)
    scaled = scaler_model.transform(assembled).cache()

    kmeans = KMeans(
        featuresCol="cluster_features",
        predictionCol="cluster",
        k=k,
        seed=42,
        maxIter=40,
    )

    model = kmeans.fit(scaled)
    clustered = model.transform(scaled).cache()

    evaluator = ClusteringEvaluator(
        featuresCol="cluster_features",
        predictionCol="cluster",
        metricName="silhouette",
        distanceMeasure="squaredEuclidean",
    )
    silhouette = evaluator.evaluate(clustered)

    print(f"K: {k}")
    print(f"Silhouette: {silhouette:.4f}")

    summary_exprs = [
        F.count("*").alias("n_posts"),
        F.avg("cqi_version_a").alias("mean_cqi"),
        F.avg("label").alias("accepted_answer_rate"),
        F.avg("post_chatgpt").alias("post_chatgpt_rate"),
        F.avg("log_reputation").alias("mean_log_reputation"),
        F.avg("code_block_count_num").alias("mean_code_blocks"),
        F.avg("log_code_len").alias("mean_log_code_len"),
        F.avg("has_code_int").alias("has_code_rate"),
        F.avg("is_parseable_int").alias("parseable_rate"),
    ]

    cluster_summary = (
        clustered
        .groupBy("cluster")
        .agg(*summary_exprs)
        .orderBy("cluster")
    )

    cluster_summary.show(truncate=False)

    metrics_df = spark.createDataFrame(
        [(run_label, start_month, end_month, int(k), float(silhouette), int(clean_df.count()))],
        ["run_label", "start_month", "end_month", "k", "silhouette", "n_rows"]
    )
    metrics_df.show(truncate=False)

    center_rows = []
    for cluster_id, center in enumerate(model.clusterCenters()):
        for feature, value in zip(clustering_features, center):
            center_rows.append((int(cluster_id), feature, float(value)))

    centers_df = spark.createDataFrame(
        center_rows,
        ["cluster", "feature", "standardized_center_value"]
    )

    sample_cols = [
        "Id", "year_month_str", "cluster", "label", "cqi_version_a",
        "experience_tier", "tag_category", "post_chatgpt",
        "log_reputation", "code_block_count_num", "log_code_len",
    ]
    sample_df = clustered.select(*[c for c in sample_cols if c in clustered.columns]).limit(200)

    metrics_df.coalesce(1).write.mode("overwrite").option("header", True).csv(s3_output_path + "metrics_csv")
    cluster_summary.coalesce(1).write.mode("overwrite").option("header", True).csv(s3_output_path + "cluster_summary_csv")
    centers_df.coalesce(1).write.mode("overwrite").option("header", True).csv(s3_output_path + "cluster_centers_csv")
    sample_df.coalesce(1).write.mode("overwrite").option("header", True).csv(s3_output_path + "cluster_sample_csv")

    print("Wrote KMeans outputs to:", s3_output_path)
    print(f"aws s3 ls {s3_output_path} --recursive --summarize")

    return model, clustered, metrics_df, cluster_summary, centers_df

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## Smoke Run / 小数据测试

This section runs a small window around the ChatGPT cutoff: `2022-10` to `2023-02`.

Run this first to confirm the data path, label construction, Spark ML pipeline, and S3 output behavior.


In [24]:
# ── Smoke: load and prepare data ─────────────────────────────────────────────
SMOKE_START_MONTH = "2022-10"
SMOKE_END_MONTH = "2023-02"

smoke_df, smoke_target_mode = load_and_prepare_ml_data(SMOKE_START_MONTH, SMOKE_END_MONTH)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Available columns:
['Id', 'PostTypeId', 'CreationDate', 'Score', 'Tags', 'OwnerUserId', 'AnswerCount', 'ClosedDate', 'AcceptedAnswerId', 'is_python', 'is_duplicate', 'code_block_count', 'primary_code_len', 'cyclomatic_complexity', 'halstead_volume', 'is_parseable', 'has_code', 'cqi_version_a', 'Reputation', 'experience_tier', 'year_month']
Rows before ML cleaning (2022-10 to 2023-02): 85,553
Partitions: 34
ML target mode: accepted_answer
Target description: has accepted answer
Clean ML rows: 84,152
+-----+-----+
|label|count|
+-----+-----+
|  0.0|49069|
|  1.0|35083|
+-----+-----+

+--------+--------------+-----+------------------+---------------+------------+------------+
|Id      |year_month_str|label|cqi_version_a     |experience_tier|tag_category|post_chatgpt|
+--------+--------------+-----+------------------+---------------+------------+------------+
|74581422|2022-11       |1.0  |0.8439677902230219|intermediate   |other       |0           |
|74445459|2022-11       |0.0  |0.2     

In [10]:
# ── Smoke: train/evaluate/write ──────────────────────────────────────────────
smoke_cv_model, smoke_predictions, smoke_metrics, smoke_confusion, smoke_coef = train_evaluate_and_write(
    df=smoke_df,
    target_mode=smoke_target_mode,
    run_label="smoke",
    start_month=SMOKE_START_MONTH,
    end_month=SMOKE_END_MONTH,
    cv_folds=2,
    param_reg=[0.01, 0.05],
    param_enet=[0.0, 1.0],
)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Target mode: accepted_answer
Numeric features: ['cqi_version_a', 'log_reputation', 'code_block_count_num', 'log_code_len', 'cyclomatic_complexity_num', 'log_halstead', 'has_code_int', 'is_parseable_int', 'is_duplicate_int', 'primary_code_len_bucket', 'post_chatgpt', 'months_since_cutoff']
Pipeline stages: ['StringIndexer', 'StringIndexer', 'OneHotEncoder', 'Bucketizer', 'VectorAssembler', 'LogisticRegression']
Train: 67,476   Test: 16,622
Fitting CrossValidator on new AWS EMR cluster (8 core nodes)...
Run label: smoke
Param grid size: 4
Test AUC     : 0.6823
Test Accuracy: 0.6374
Test F1      : 0.6237
Best regParam       : 0.01
Best elasticNetParam: 0.0
Intercept           : -2.2423
+-----+----------+----------------------------------------+------------------+---------------+-------------+
|label|prediction|probability                             |cqi_version_a     |experience_tier|tag_category |
+-----+----------+----------------------------------------+------------------+------------

In [25]:
# ── Smoke: unsupervised KMeans ───────────────────────────────────────────────
# If smoke_df does not exist anymore, rerun the Smoke load/prep cell first.

smoke_kmeans_model, smoke_clustered, smoke_kmeans_metrics, smoke_cluster_summary, smoke_cluster_centers = run_kmeans_and_write(
    df=smoke_df,
    run_label="smoke",
    start_month=SMOKE_START_MONTH,
    end_month=SMOKE_END_MONTH,
    k=4,
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

KMeans run label: smoke
Rows for clustering: 84,152
Features: ['cqi_version_a', 'log_reputation', 'code_block_count_num', 'log_code_len', 'cyclomatic_complexity_num', 'log_halstead', 'has_code_int', 'is_parseable_int', 'is_duplicate_int', 'post_chatgpt', 'months_since_cutoff']
K: 4
Silhouette: 0.3389
+-------+-------+-------------------+--------------------+------------------+-------------------+------------------+------------------+-------------+--------------+
|cluster|n_posts|mean_cqi           |accepted_answer_rate|post_chatgpt_rate |mean_log_reputation|mean_code_blocks  |mean_log_code_len |has_code_rate|parseable_rate|
+-------+-------+-------------------+--------------------+------------------+-------------------+------------------+------------------+-------------+--------------+
|0      |31917  |0.7289938429973595 |0.45392737412664097 |0.5505216655700724|4.18172060649517   |3.226587711877683 |5.5742532493816555|1.0          |1.0           |
|1      |8357   |0.30679416960844413|0

## Full Run / 全量窗口

This section runs the final analysis window: `2019-01` to `2025-03`.

Only run this after the smoke section succeeds.


In [19]:
# ── Full: load and prepare data ──────────────────────────────────────────────
FULL_START_MONTH = "2019-01"
FULL_END_MONTH = "2025-03"

full_df, full_target_mode = load_and_prepare_ml_data(FULL_START_MONTH, FULL_END_MONTH)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Available columns:
['Id', 'PostTypeId', 'CreationDate', 'Score', 'Tags', 'OwnerUserId', 'AnswerCount', 'ClosedDate', 'AcceptedAnswerId', 'is_python', 'is_duplicate', 'code_block_count', 'primary_code_len', 'cyclomatic_complexity', 'halstead_volume', 'is_parseable', 'has_code', 'cqi_version_a', 'Reputation', 'experience_tier', 'year_month']
Rows before ML cleaning (2019-01 to 2025-03): 1,263,344
Partitions: 83
ML target mode: accepted_answer
Target description: has accepted answer
Clean ML rows: 1,234,826
+-----+------+
|label| count|
+-----+------+
|  0.0|679634|
|  1.0|555192|
+-----+------+

+--------+--------------+-----+-------------------+---------------+------------+------------+
|Id      |year_month_str|label|cqi_version_a      |experience_tier|tag_category|post_chatgpt|
+--------+--------------+-----+-------------------+---------------+------------+------------+
|61036050|2020-04       |0.0  |0.6158883083359672 |intermediate   |other       |0           |
|61139640|2020-04      

In [22]:
# ── Full: train/evaluate/write ───────────────────────────────────────────────
full_cv_model, full_predictions, full_metrics, full_confusion, full_coef = train_evaluate_and_write(
    df=full_df,
    target_mode=full_target_mode,
    run_label="full",
    start_month=FULL_START_MONTH,
    end_month=FULL_END_MONTH,
    cv_folds=5,
    param_reg=[0.0, 0.01, 0.03, 0.05, 0.1],
    param_enet=[0.0, 0.5, 1.0],
)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Target mode: accepted_answer
Numeric features: ['cqi_version_a', 'log_reputation', 'code_block_count_num', 'log_code_len', 'cyclomatic_complexity_num', 'log_halstead', 'has_code_int', 'is_parseable_int', 'is_duplicate_int', 'primary_code_len_bucket', 'post_chatgpt', 'months_since_cutoff']
Pipeline stages: ['StringIndexer', 'StringIndexer', 'OneHotEncoder', 'Bucketizer', 'VectorAssembler', 'LogisticRegression']
Train: 987,499   Test: 247,327
Fitting CrossValidator on new AWS EMR cluster (8 core nodes)...
Run label: full
Param grid size: 15
Test AUC     : 0.6881
Test Accuracy: 0.6382
Test F1      : 0.6338
Best regParam       : 0.0
Best elasticNetParam: 0.0
Intercept           : -0.7308
+-----+----------+----------------------------------------+-------------------+---------------+-------------+
|label|prediction|probability                             |cqi_version_a      |experience_tier|tag_category |
+-----+----------+----------------------------------------+-------------------+--------

In [26]:
# ── Full: unsupervised KMeans ────────────────────────────────────────────────
# If full_df does not exist anymore, rerun the Full load/prep cell first.

full_kmeans_model, full_clustered, full_kmeans_metrics, full_cluster_summary, full_cluster_centers = run_kmeans_and_write(
    df=full_df,
    run_label="full",
    start_month=FULL_START_MONTH,
    end_month=FULL_END_MONTH,
    k=4,
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

KMeans run label: full
Rows for clustering: 1,234,826
Features: ['cqi_version_a', 'log_reputation', 'code_block_count_num', 'log_code_len', 'cyclomatic_complexity_num', 'log_halstead', 'has_code_int', 'is_parseable_int', 'is_duplicate_int', 'post_chatgpt', 'months_since_cutoff']
K: 4
Silhouette: 0.4744
+-------+-------+-------------------+--------------------+-------------------+-------------------+------------------+------------------+------------------+-------------------+
|cluster|n_posts|mean_cqi           |accepted_answer_rate|post_chatgpt_rate  |mean_log_reputation|mean_code_blocks  |mean_log_code_len |has_code_rate     |parseable_rate     |
+-------+-------+-------------------+--------------------+-------------------+-------------------+------------------+------------------+------------------+-------------------+
|0      |777861 |0.5176831078462083 |0.46976259254545477 |0.1569457782303008 |4.63816739675146   |3.4574094857564526|5.8046223730524735|1.0               |0.43901031161